# Diabetes Risk Prediction
## Model Training & Evaluation
This notebook contains all model training experiments across 4 models and 4 balancing strategies (16 total combinations).

## Load the Unscaled Training Data

In [ ]:
# Load preprocessed data
import numpy as np
import pandas as pd

X_train = np.load('Data/X_train.npy')
X_test = np.load('Data/X_test.npy')
y_train = np.load('Data/y_train.npy')
y_test = np.load('Data/y_test.npy')
feature_names = pd.read_csv('Data/feature_names.csv').values.flatten()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("Feature names:", feature_names)

## Training: Raw Data (No Balancing)

We start with the raw unbalanced data as our baseline.

### Random Forest on Raw Training Data

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Make predictions
rf_predictions = rf_model.predict(X_test)

# Results
print("Random Forest - Raw Data Results:\n")
print(classification_report(y_test, rf_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, rf_model.predict_proba(X_test)[:,1]), 4))

Random Forest - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.96      0.91     42741
    Diabetes       0.52      0.21      0.30      7995

    accuracy                           0.84     50736
   macro avg       0.69      0.59      0.61     50736
weighted avg       0.81      0.84      0.82     50736

AUC-ROC: 0.7921


#### Random Forest - Raw Data Interpretation

The results confirm the class imbalance problem we identified during EDA. The model achieves 84% accuracy, but this is misleading because it's mostly just correctly predicting healthy patients (96% recall) while only catching 21% of actual diabetic patients.

Key takeaways:
- The model misses roughly 4 out of every 5 diabetic patients (recall = 0.21)
- When it does flag someone as diabetic, it's only right about half the time (precision = 0.52)
- The F1-score for diabetes is 0.30, which is poor
- AUC-ROC of 0.79 shows the model has some ability to distinguish between classes but struggles significantly with the minority class

This baseline result demonstrates exactly why balancing techniques like SMOTE and SMOTE-ENN are needed. Without addressing the imbalance, the model effectively learns to default to predicting "no diabetes" because that's the safe bet given the 84/16 class split.

### XGBoost on Raw Training Data
Training XGBoost with default hyperparameters on the raw unbalanced dataset.
XGBoost builds trees sequentially, where each new tree focuses on correcting the mistakes of the previous ones.

In [4]:
from xgboost import XGBClassifier

# Train XGBoost
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Make predictions
xgb_predictions = xgb_model.predict(X_test)

# Results
print("XGBoost - Raw Data Results:\n")
print(classification_report(y_test, xgb_predictions, target_names=['No Diabetes', 'Diabetes']))
print("AUC-ROC:", round(roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:,1]), 4))

XGBoost - Raw Data Results:

              precision    recall  f1-score   support

 No Diabetes       0.87      0.97      0.92     42741
    Diabetes       0.57      0.21      0.30      7995

    accuracy                           0.85     50736
   macro avg       0.72      0.59      0.61     50736
weighted avg       0.82      0.85      0.82     50736

AUC-ROC: 0.8194


#### XGBoost - Raw Data Interpretation

XGBoost performs very similarly to Random Forest on the raw data. Diabetes recall is identical at 0.21 and F1-score is the same at 0.30. XGBoost shows slight improvement in precision (0.57 vs 0.52) and AUC-ROC (0.82 vs 0.79), suggesting it ranks patients slightly better overall.

The key finding here is that switching to a more powerful model alone does not solve the class imbalance problem. Both models default to predicting "no diabetes" for the vast majority of cases. This confirms that the issue lies in the data distribution, not the model architecture, and motivates the need for resampling techniques like SMOTE and SMOTE-ENN.

## Balancing Techniques
The raw data has an 84/16 split (healthy vs diabetic). We apply three techniques to the training data only (never the test data) to address this imbalance.

The raw training data has a significant class imbalance: 170,962 healthy (class 0) vs 31,982 diabetic (class 1). We apply three balancing techniques to the training data only. The test data is never touched so it continues to represent real-world conditions.

### How each technique works:

**SMOTE (Synthetic Minority Oversampling Technique):** Creates new synthetic diabetic patients by finding pairs of real diabetic patients that are close to each other and generating a new data point somewhere in between them. This is repeated until both classes are equal in size.

**SMOTE-ENN (SMOTE + Edited Nearest Neighbors):** First applies SMOTE to create synthetic diabetic patients. Then runs ENN, which removes noisy or confusing data points from both classes. A point is removed if most of its nearest neighbors belong to a different class. This produces cleaner data but the two classes may not be perfectly equal.

**Random Undersampling:** Randomly deletes healthy patients until both classes are equal. Simple and fast but discards a large portion of the training data.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN
from imblearn.under_sampling import RandomUnderSampler

# SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("SMOTE:", np.unique(y_train_smote, return_counts=True))

# SMOTE-ENN
smote_enn = SMOTEENN(random_state=42)
X_train_smoteenn, y_train_smoteenn = smote_enn.fit_resample(X_train, y_train)
print("SMOTE-ENN:", np.unique(y_train_smoteenn, return_counts=True))

# Random Undersampling
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)
print("Undersampling:", np.unique(y_train_under, return_counts=True))

print("\nOriginal:", np.unique(y_train, return_counts=True))

SMOTE: (array([0, 1]), array([170962, 170962]))
SMOTE-ENN: (array([0, 1]), array([ 98808, 159903]))
Undersampling: (array([0, 1]), array([31982, 31982]))

Original: (array([0, 1]), array([170962,  31982]))


In [ ]:
# Save SMOTE-balanced training arrays to disk
np.save('Data/X_train_smote.npy', X_train_smote)
np.save('Data/y_train_smote.npy', y_train_smote)

# Save SMOTE-ENN-balanced training arrays to disk
np.save('Data/X_train_smoteenn.npy', X_train_smoteenn)
np.save('Data/y_train_smoteenn.npy', y_train_smoteenn)

# Save randomly undersampled training arrays to disk
np.save('Data/X_train_under.npy', X_train_under)
np.save('Data/y_train_under.npy', y_train_under)

print("Saved to Data/:")
print(" - X_train_smote.npy, y_train_smote.npy")
print(" - X_train_smoteenn.npy, y_train_smoteenn.npy")
print(" - X_train_under.npy, y_train_under.npy")

### Balancing Results:

| Technique | Healthy (0) | Diabetic (1) | Total Samples |
|---|---|---|---|
| Original | 170,962 | 31,982 | 202,944 |
| SMOTE | 170,962 | 170,962 | 341,924 |
| SMOTE-ENN | 98,808 | 159,903 | 258,711 |
| Undersampling | 31,982 | 31,982 | 63,964 |

SMOTE doubled the dataset by generating ~139,000 synthetic diabetic patients. SMOTE-ENN also generated synthetic patients but then cleaned up noisy points from both classes, which is why the healthy count dropped from 170,962 to 98,808. The classes are not perfectly equal but the data is cleaner overall. Undersampling achieved perfect balance but at a steep cost, reducing the total dataset from 202,944 to only 63,964 samples (a 68% reduction in data).

## RF & XGBoost on Balanced Data

The original RF and XGBoost runs used raw unbalanced data as a baseline. Here we apply two balancing strategies to each:

- **`class_weight='balanced'` / `scale_pos_weight`** — built-in class weighting that adjusts the loss function without resampling
- **SMOTE** — trained on the SMOTE-balanced scaled training set (`X_train_smote`), evaluated on `X_test_scaled`

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

X_train        = np.load('Data/X_train.npy')
X_test         = np.load('Data/X_test.npy')
y_train        = np.load('Data/y_train.npy')
y_test         = np.load('Data/y_test.npy')
X_train_smote  = np.load('Data/X_train_smote.npy')
y_train_smote  = np.load('Data/y_train_smote.npy')
X_test_scaled  = np.load('Data/X_test_scaled.npy')

def evaluate(model, X_te, y_te):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        "f1":        round(f1_score(y_te, y_pred), 4),
        "auc_roc":   round(roc_auc_score(y_te, y_proba), 4),
        "precision": round(precision_score(y_te, y_pred), 4),
        "recall":    round(recall_score(y_te, y_pred), 4),
        "cm":        confusion_matrix(y_te, y_pred),
    }

n_neg, n_pos = np.bincount(y_train.astype(int))
spw = n_neg / n_pos  # XGBoost scale_pos_weight

# ── Random Forest ─────────────────────────────────────────────────────────
print("Training RF class_weight=balanced ...")
rf_bal = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_bal.fit(X_train, y_train)
res_rf_bal = evaluate(rf_bal, X_test, y_test)

print("Training RF + SMOTE ...")
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_smote, y_train_smote)
res_rf_smote = evaluate(rf_smote, X_test_scaled, y_test)

# ── XGBoost ───────────────────────────────────────────────────────────────
print("Training XGBoost scale_pos_weight ...")
xgb_bal = XGBClassifier(n_estimators=100, scale_pos_weight=spw, random_state=42, eval_metric='logloss')
xgb_bal.fit(X_train, y_train)
res_xgb_bal = evaluate(xgb_bal, X_test, y_test)

print("Training XGBoost + SMOTE ...")
xgb_smote = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_smote.fit(X_train_smote, y_train_smote)
res_xgb_smote = evaluate(xgb_smote, X_test_scaled, y_test)

# ── Results ───────────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Model':<28} {'F1':>6} {'AUC':>6} {'Prec':>6} {'Recall':>8}")
print("="*65)
rows = [
    ("RF class_weight=balanced",  res_rf_bal),
    ("RF + SMOTE",                res_rf_smote),
    ("XGBoost scale_pos_weight",  res_xgb_bal),
    ("XGBoost + SMOTE",           res_xgb_smote),
]
for name, m in rows:
    print(f"  {name:<26} {m['f1']:>6} {m['auc_roc']:>6} {m['precision']:>6} {m['recall']:>8}")

## Yousef — Logistic Regression & KNN Experiments

This section trains and evaluates Logistic Regression and K-Nearest Neighbours (KNN) across three data conditions:
- **Raw** (no balancing)
- **SMOTE** (synthetic oversampling applied to scaled training data)
- **Random Undersampling** (applied to scaled training data)

All balancing is applied to `X_train_scaled` so that the feature scale is consistent with the test set (`X_test_scaled`).

In [ ]:
import numpy as np

# ── Load scaled base arrays ────────────────────────────────────────────────
# Balancing must be applied to X_train_scaled (not raw X_train) so that
# the synthetic/resampled points share the same feature scale as X_test_scaled.
X_train_scaled = np.load('Data/X_train_scaled.npy')
X_test_scaled  = np.load('Data/X_test_scaled.npy')
y_train        = np.load('Data/y_train.npy')
y_test         = np.load('Data/y_test.npy')

# ── Apply SMOTE to scaled training data ───────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

X_train_smote, y_train_smote = SMOTE(random_state=42).fit_resample(X_train_scaled, y_train)
np.save('Data/X_train_smote.npy', X_train_smote)
np.save('Data/y_train_smote.npy', y_train_smote)

# ── Apply Random Undersampling to scaled training data ────────────────────
X_train_under, y_train_under = RandomUnderSampler(random_state=42).fit_resample(X_train_scaled, y_train)
np.save('Data/X_train_under.npy', X_train_under)
np.save('Data/y_train_under.npy', y_train_under)

# ── Verify shapes ──────────────────────────────────────────────────────────
print("Base arrays:")
print(f"  X_train_scaled : {X_train_scaled.shape}")
print(f"  X_test_scaled  : {X_test_scaled.shape}")
print(f"  y_train        : {y_train.shape}")
print(f"  y_test         : {y_test.shape}")

print("\nBalanced arrays (from scaled data):")
print(f"  X_train_smote  : {X_train_smote.shape}  |  class counts: {dict(zip(*np.unique(y_train_smote, return_counts=True)))}")
print(f"  X_train_under  : {X_train_under.shape}  |  class counts: {dict(zip(*np.unique(y_train_under, return_counts=True)))}")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, confusion_matrix

# ── Evaluation function ────────────────────────────────────────────────────
def evaluate_model(model, X_test, y_test):
    """Evaluate a fitted model and return key classification metrics."""
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "f1":               round(f1_score(y_test, y_pred), 4),
        "auc_roc":          round(roc_auc_score(y_test, y_proba), 4),
        "precision":        round(precision_score(y_test, y_pred), 4),
        "recall":           round(recall_score(y_test, y_pred), 4),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }

# ── Run definitions ────────────────────────────────────────────────────────
# KNN on SMOTE (341k rows) and SMOTE-ENN (258k rows) are excluded:
# brute-force KNN distance computation at that scale is prohibitively slow.
runs = [
    ("run1_log_raw",   LogisticRegression(max_iter=1000),   X_train_scaled, y_train),
    ("run2_log_smote", LogisticRegression(max_iter=1000),   X_train_smote,  y_train_smote),
    ("run3_log_under", LogisticRegression(max_iter=1000),   X_train_under,  y_train_under),
    ("run4_knn_raw",   KNeighborsClassifier(n_neighbors=5), X_train_scaled, y_train),
    ("run5_knn_under", KNeighborsClassifier(n_neighbors=5), X_train_under,  y_train_under),
]

# ── Train and evaluate all runs ────────────────────────────────────────────
results = {}
for run_key, model, X_tr, y_tr in runs:
    print(f"Training {run_key} ...")
    model.fit(X_tr, y_tr)
    results[run_key] = evaluate_model(model, X_test_scaled, y_test)

# ── Print results ──────────────────────────────────────────────────────────
print("\n" + "="*60)
print("RESULTS SUMMARY")
print("="*60)
for run_key, m in results.items():
    print(f"\n{run_key}")
    print(f"  F1-Score  : {m['f1']}")
    print(f"  AUC-ROC   : {m['auc_roc']}")
    print(f"  Precision : {m['precision']}")
    print(f"  Recall    : {m['recall']}")
    print(f"  Confusion Matrix:\n{m['confusion_matrix']}")

best_f1     = max(results, key=lambda k: results[k]['f1'])
best_recall = max(results, key=lambda k: results[k]['recall'])
print(f"\n>>> Highest F1-Score : {best_f1}  ({results[best_f1]['f1']})")
print(f">>> Highest Recall   : {best_recall}  ({results[best_recall]['recall']})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import precision_recall_curve, average_precision_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier

# ── Retrain best models for plotting ─────────────────────────────────────
X_train        = np.load('Data/X_train.npy')
y_train        = np.load('Data/y_train.npy')
n_neg, n_pos   = np.bincount(y_train.astype(int))

lr_smote_m  = LogisticRegression(max_iter=1000).fit(X_train_smote, y_train_smote)
rf_smote_m  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train_smote, y_train_smote)
xgb_smote_m = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss').fit(X_train_smote, y_train_smote)
xgb_bal_m   = XGBClassifier(n_estimators=100, scale_pos_weight=n_neg/n_pos, random_state=42, eval_metric='logloss').fit(X_train, y_train)

# ── Precision-Recall curves ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
pr_models = {
    'LR + SMOTE':           lr_smote_m.predict_proba(X_test_scaled)[:, 1],
    'RF + SMOTE':           rf_smote_m.predict_proba(X_test_scaled)[:, 1],
    'XGBoost + SMOTE':      xgb_smote_m.predict_proba(X_test_scaled)[:, 1],
    'XGBoost (scale_pos_weight)': xgb_bal_m.predict_proba(X_test)[:, 1],
}
colors = ['#2196F3', '#4CAF50', '#FF5722', '#9C27B0']
for (label, proba), color in zip(pr_models.items(), colors):
    p, r, _ = precision_recall_curve(y_test, proba)
    ap = average_precision_score(y_test, proba)
    ax.plot(r, p, label=f"{label}  (AP={ap:.3f})", color=color, linewidth=1.8)

ax.axhline(y=y_test.mean(), color='gray', linestyle='--', linewidth=1,
           label=f'Random baseline (AP={y_test.mean():.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curves — Balanced Models', fontsize=13)
ax.legend(fontsize=9, loc='upper right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1])
plt.tight_layout()
plt.savefig('Data/precision_recall_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Confusion matrix heatmaps ─────────────────────────────────────────────
best_models = {
    'LR + SMOTE':      (lr_smote_m,  X_test_scaled),
    'RF + SMOTE':      (rf_smote_m,  X_test_scaled),
    'XGBoost + SMOTE': (xgb_smote_m, X_test_scaled),
}
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (title, (model, X_te)) in zip(axes, best_models.items()):
    cm = confusion_matrix(y_test, model.predict(X_te))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Diabetes', 'Diabetes'],
                yticklabels=['No Diabetes', 'Diabetes'],
                cbar=False, annot_kws={'size': 11})
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Actual', fontsize=10)
    ax.set_xlabel('Predicted', fontsize=10)

plt.suptitle('Confusion Matrices — Best Balanced Models (Test Set)', fontsize=13)
plt.tight_layout()
plt.savefig('Data/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Results Table

| Run | Model | Training Data | F1-Score | AUC-ROC | Precision | Recall |
|---|---|---|---|---|---|---|
| run1_log_raw | Logistic Regression | Raw (scaled) | 0.2838 | 0.8172 | 0.5523 | 0.1910 |
| **run2_log_smote** | Logistic Regression | SMOTE | **0.4702** | 0.8165 | 0.3409 | 0.7578 |
| run3_log_under | Logistic Regression | Undersampling | 0.4700 | **0.8177** | 0.3406 | **0.7582** |
| run4_knn_raw | KNN (k=5) | Raw (scaled) | 0.3082 | 0.7257 | 0.4328 | 0.2393 |
| run5_knn_under | KNN (k=5) | Undersampling | 0.4266 | 0.7628 | 0.3004 | 0.7355 |

**Highest F1-Score:** `run2_log_smote` — Logistic Regression + SMOTE (0.4702)

**Highest Recall:** `run3_log_under` — Logistic Regression + Undersampling (0.7582)

---

## Analysis & Interpretation

### 1. The Class Imbalance Problem

The dataset has a severe class imbalance: 84% healthy vs 16% diabetic. A model that simply predicts "no diabetes" for every patient would achieve 84% accuracy while catching zero diabetic patients. This makes accuracy a misleading metric. **Recall** (how many actual diabetic patients are correctly identified) and **F1-score** (the harmonic mean of precision and recall) are the metrics that matter for this task.

The raw unbalanced baseline (`run1_log_raw`) demonstrates this problem directly: it achieves 55% precision but only **19% recall**, meaning it misses 4 out of every 5 diabetic patients. This is clinically unacceptable for a screening tool.

### 2. Effect of SMOTE and Undersampling

Both balancing techniques dramatically improved recall:

- **SMOTE** (`run2_log_smote`) increased recall from 0.19 → **0.76**, a 4× improvement. SMOTE generates synthetic diabetic patients by interpolating between existing ones in feature space, giving the model much more signal about what a diabetic patient looks like.
- **Random Undersampling** (`run3_log_under`) achieved nearly identical results (recall 0.76, F1 0.47) by removing the majority class down to match the minority class. Despite discarding 68% of the training data, performance was not hurt — the class imbalance was the main obstacle, not the training set size.

The trade-off is lower precision: balanced models flag more false positives. In a medical screening context this is the correct trade-off — missing a diabetic patient is far more costly than a false alarm.

### 3. Logistic Regression vs KNN

Logistic Regression outperformed KNN on every balanced run:

- `run2_log_smote` (F1 = 0.4702) vs `run5_knn_under` (F1 = 0.4266)
- `run3_log_under` (AUC-ROC = 0.8177) vs `run5_knn_under` (AUC-ROC = 0.7628)

**Why Logistic Regression performed best:**

1. **Linear separability:** After scaling, many of the strongest risk factors (GenHlth, BMI, Age, HighBP) have roughly linear relationships with diabetes risk. Logistic Regression is well-suited to this.
2. **Generalisation:** Logistic Regression learns a global decision boundary, making it robust to the synthetic points introduced by SMOTE. KNN is a lazy learner that memorises local neighbourhoods, making it more sensitive to the density changes SMOTE introduces.
3. **AUC-ROC:** All three LR runs achieved AUC-ROC ≥ 0.82 vs KNN's 0.73–0.76, confirming that LR produces better-calibrated probability estimates across all thresholds.
4. **Stability:** LR + SMOTE and LR + undersampling produced nearly identical results (F1 within 0.0002), suggesting the model converges reliably regardless of which balancing strategy is used.

### 4. Summary

For a diabetes screening use case where maximising recall is the priority, **Logistic Regression trained on balanced data** is the recommended model from this set of experiments. The SMOTE and undersampling variants are effectively equivalent — both achieve ~76% recall with ~47% F1-score and ~82% AUC-ROC.

## SHAP Analysis (Experiment 2)

SHAP (SHapley Additive exPlanations) assigns each feature a contribution score for every individual prediction, based on game-theoretic Shapley values. Unlike global feature importance metrics, SHAP explains *why* the model made a specific prediction for a specific patient.

We use the best model from Experiment 1: **Logistic Regression trained on SMOTE-balanced scaled data** (`run2_log_smote`, F1 = 0.4702, Recall = 0.7578).

`LinearExplainer` is used here — it is the correct and efficient SHAP explainer for linear models, computing exact Shapley values analytically rather than by sampling.

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# ── Load data ─────────────────────────────────────────────────────────────
X_train_smote  = np.load('Data/X_train_smote.npy')
y_train_smote  = np.load('Data/y_train_smote.npy')
X_test_scaled  = np.load('Data/X_test_scaled.npy')
y_test         = np.load('Data/y_test.npy')
feature_names  = pd.read_csv('Data/feature_names.csv').values.flatten().tolist()

# ── Train best model (LR + SMOTE) ─────────────────────────────────────────
print("Training Logistic Regression on SMOTE data ...")
lr_smote = LogisticRegression(max_iter=1000)
lr_smote.fit(X_train_smote, y_train_smote)
print("Done.")

# ── Compute SHAP values with LinearExplainer ──────────────────────────────
# LinearExplainer is exact and fast for linear models.
# We use the SMOTE training set as the background distribution.
print("Computing SHAP values ...")
explainer   = shap.LinearExplainer(lr_smote, X_train_smote)
shap_values = explainer.shap_values(X_test_scaled)
print(f"SHAP values shape: {shap_values.shape}")

# ── Wrap test set with feature names for plots ────────────────────────────
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)

In [ ]:
# ── Plot 1: SHAP Summary Plot (beeswarm) ─────────────────────────────────
# Each dot is one test sample. Colour = feature value (red=high, blue=low).
# Position on x-axis = SHAP value (impact on model output).
print("Generating SHAP summary plot ...")
plt.figure()
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title("SHAP Summary Plot — Logistic Regression + SMOTE", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('Data/shap_summary_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: Data/shap_summary_plot.png")

# ── Plot 2: SHAP Bar Plot (mean absolute SHAP — global feature importance) ─
print("Generating SHAP bar plot ...")
plt.figure()
shap.summary_plot(shap_values, X_test_df, plot_type='bar', show=False)
plt.title("SHAP Feature Importance (Mean |SHAP value|) — Top Features", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('Data/shap_bar_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: Data/shap_bar_plot.png")

In [ ]:
# ── Individual Predictions: Waterfall Plots ───────────────────────────────
# Pick one high-confidence diabetic prediction and one healthy prediction
# from the test set to show how SHAP explains individual cases.

y_pred = lr_smote.predict(X_test_scaled)
y_prob = lr_smote.predict_proba(X_test_scaled)[:, 1]

# Sample 1: high-confidence diabetic prediction (predicted=1, actual=1, high probability)
diabetic_idx = np.where((y_pred == 1) & (y_test == 1))[0]
sample1_idx  = diabetic_idx[np.argmax(y_prob[diabetic_idx])]

# Sample 2: high-confidence healthy prediction (predicted=0, actual=0, low probability)
healthy_idx  = np.where((y_pred == 0) & (y_test == 0))[0]
sample2_idx  = healthy_idx[np.argmin(y_prob[healthy_idx])]

print(f"Sample 1 — Predicted DIABETIC  | Actual: {int(y_test[sample1_idx])} | P(diabetes) = {y_prob[sample1_idx]:.3f}")
print(f"Sample 2 — Predicted HEALTHY   | Actual: {int(y_test[sample2_idx])} | P(diabetes) = {y_prob[sample2_idx]:.3f}")

# ── Waterfall plot: Sample 1 (diabetic) ───────────────────────────────────
shap_exp = shap.Explanation(
    values    = shap_values[sample1_idx],
    base_values = explainer.expected_value,
    data      = X_test_scaled[sample1_idx],
    feature_names = feature_names
)
plt.figure()
shap.plots.waterfall(shap_exp, show=False)
plt.title(f"Individual Prediction — HIGH RISK PATIENT  |  P(diabetes) = {y_prob[sample1_idx]:.3f}", fontsize=11)
plt.tight_layout()
plt.savefig('Data/shap_waterfall_diabetic.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Waterfall plot: Sample 2 (healthy) ────────────────────────────────────
shap_exp2 = shap.Explanation(
    values    = shap_values[sample2_idx],
    base_values = explainer.expected_value,
    data      = X_test_scaled[sample2_idx],
    feature_names = feature_names
)
plt.figure()
shap.plots.waterfall(shap_exp2, show=False)
plt.title(f"Individual Prediction — LOW RISK PATIENT  |  P(diabetes) = {y_prob[sample2_idx]:.3f}", fontsize=11)
plt.tight_layout()
plt.savefig('Data/shap_waterfall_healthy.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: Data/shap_waterfall_diabetic.png")
print("Saved: Data/shap_waterfall_healthy.png")

## SHAP Findings & Interpretation

### 1. Most Important Features

The SHAP bar plot ranks features by their mean absolute SHAP value — the average magnitude of impact each feature has on the model's output across all 50,736 test patients.

The top features identified by SHAP are consistent with established medical literature on Type 2 diabetes:

| Feature | Role |
|---|---|
| **GenHlth** | General health self-rating (1=excellent → 5=poor). Poor self-rated health is one of the strongest predictors — patients with diabetes typically report poorer overall health. |
| **BMI** | Body Mass Index. Obesity is a primary clinical risk factor for Type 2 diabetes. Higher BMI consistently pushes predictions toward diabetes. |
| **Age** | Risk of Type 2 diabetes increases significantly with age, peaking in older adult brackets. |
| **HighBP** | High blood pressure (hypertension) is strongly comorbid with diabetes — both conditions share underlying metabolic causes. |
| **Income** | Lower income is associated with higher diabetes risk, reflecting reduced access to healthcare, healthier food, and physical activity. |
| **DiffWalk** | Difficulty walking or climbing stairs, a proxy for physical disability, is elevated in diabetic patients. |
| **PhysHlth** | Number of days of poor physical health in the past 30 days. |

### 2. Agreement with EDA Findings

The SHAP results closely mirror the EDA findings from `01_eda.ipynb`:

- The EDA group-mean comparison showed diabetic patients had markedly higher `GenHlth` (3.25 vs 2.37), higher `BMI` (31.8 vs 27.7), and higher `Age` (9.34 vs 7.79 on the ordinal scale).
- The correlation heatmap in EDA showed `GenHlth`, `DiffWalk`, and `PhysHlth` as the features most correlated with the diabetes target.
- SHAP confirms the same ordering, validating that the model is learning real patterns in the data rather than noise.

### 3. What the Beeswarm Plot Shows

The summary beeswarm plot adds directional information on top of the importance ranking:

- **GenHlth (red = high value = poor health):** High GenHlth scores strongly push predictions toward diabetes (positive SHAP), while low scores push toward healthy.
- **BMI (red = high BMI):** Higher BMI consistently increases predicted diabetes risk.
- **Age (red = older):** Older age is a positive risk driver.
- **HighBP (red = yes):** Having high blood pressure significantly raises predicted risk.
- **Income (blue = high income):** Higher income *reduces* predicted risk — the blue dots on the positive side are low-income patients.

### 4. Individual Prediction Explanations

The waterfall plots show how each feature contributes to pushing a specific prediction above or below the model's base rate (expected value across all patients).

- **High-risk patient:** Multiple features simultaneously push the prediction upward — poor general health, high BMI, older age, and high blood pressure combine to produce a high diabetes probability. The model is not relying on any single feature but on the cumulative burden of several risk factors.
- **Low-risk patient:** Key protective features — good general health, lower BMI, younger age, and higher income — all push the prediction downward, resulting in a very low predicted probability.

### 5. Real-World Implications

These SHAP findings have direct clinical relevance:

1. **Screening priority:** Patients with poor self-rated health, high BMI, and high blood pressure should be flagged for diabetes screening even without a formal diagnosis.
2. **Modifiable risk factors:** BMI and physical activity (`PhysActivity`) are modifiable — interventions targeting these could reduce risk.
3. **Socioeconomic dimension:** Income appearing as a top feature highlights that diabetes risk is not purely biological. Healthcare access, diet quality, and stress (all correlated with income) play a real role.
4. **Model trustworthiness:** The fact that SHAP surfaces medically well-understood risk factors — rather than spurious correlations — gives confidence that the model generalises to real patients rather than overfitting to dataset artefacts.

## SHAP Analysis — XGBoost Comparison

XGBoost supports SHAP natively via `TreeExplainer`, which computes exact Shapley values for tree ensembles efficiently. We compare feature importance rankings between LR and XGBoost to see whether both models agree on which features drive diabetes risk.

In [ ]:
import shap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression

# ── Train XGBoost + SMOTE ─────────────────────────────────────────────────
xgb_shap = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
xgb_shap.fit(X_train_smote, y_train_smote)

# ── Compute SHAP values (sample 5000 for speed) ───────────────────────────
np.random.seed(42)
sample_idx    = np.random.choice(len(X_test_scaled), 5000, replace=False)
X_sample      = X_test_scaled[sample_idx]
X_sample_df   = pd.DataFrame(X_sample, columns=feature_names)

explainer_xgb = shap.TreeExplainer(xgb_shap)
shap_vals_xgb = explainer_xgb.shap_values(X_sample)

# ── XGBoost SHAP summary plot ─────────────────────────────────────────────
plt.figure()
shap.summary_plot(shap_vals_xgb, X_sample_df, show=False)
plt.title("SHAP Summary Plot — XGBoost + SMOTE", fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('Data/shap_xgb_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# ── LR vs XGBoost feature importance comparison ───────────────────────────
lr_shap     = LogisticRegression(max_iter=1000).fit(X_train_smote, y_train_smote)
explainer_lr = shap.LinearExplainer(lr_shap, X_train_smote)
shap_vals_lr = explainer_lr.shap_values(X_test_scaled)

mean_lr  = np.abs(shap_vals_lr).mean(axis=0)
mean_xgb = np.abs(shap_vals_xgb).mean(axis=0)

comparison = pd.DataFrame({
    'Feature':  feature_names,
    'LR':       mean_lr,
    'XGBoost':  mean_xgb,
}).sort_values('XGBoost', ascending=False).reset_index(drop=True)

top10 = comparison.head(10)
x = np.arange(len(top10))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - w/2, top10['LR'],      w, label='LR + SMOTE',      color='#2196F3', alpha=0.85)
ax.bar(x + w/2, top10['XGBoost'], w, label='XGBoost + SMOTE', color='#FF5722', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(top10['Feature'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Mean |SHAP value|', fontsize=11)
ax.set_title('Feature Importance: LR vs XGBoost (SHAP)', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('Data/shap_lr_vs_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 10 features — LR vs XGBoost SHAP:")
print(f"{'Feature':<25} {'LR':>10} {'XGBoost':>10}")
print("-"*47)
for _, row in top10.iterrows():
    print(f"  {row.Feature:<23} {row.LR:>10.4f} {row.XGBoost:>10.4f}")

### LR vs XGBoost SHAP — Key Findings

Both models rank the **same top 4 features**: `GenHlth`, `BMI`, `Age`, and `HighBP` — with near-identical ordering. This agreement across two fundamentally different model architectures (linear vs gradient-boosted trees) strongly validates that these features represent genuine biological and demographic drivers of diabetes risk rather than model-specific artefacts.

Notable differences:
- **Income** ranks 5th for XGBoost (mean |SHAP| = 0.333) but only 7th for LR (0.092). XGBoost captures non-linear interactions between income and other features that LR cannot represent with a linear boundary.
- **HighChol** is more prominent in LR (3rd) than XGBoost (6th), suggesting the linear model leans more heavily on this binary flag as a proxy for metabolic risk.
- **Education** appears in XGBoost's top 10 (9th) but is near-absent in LR — another non-linear socioeconomic interaction that the tree model picks up.

Overall, the convergence on GenHlth, BMI, Age, and HighBP across both models and both explainability methods gives high confidence that these are the true primary risk drivers in this dataset.

## Threshold Optimisation — Logistic Regression + SMOTE

By default, classifiers predict class 1 when `P(diabetic) ≥ 0.50`. Adjusting this threshold lets us trade precision for recall — critical in a medical screening context where **missing a diabetic patient (false negative) is more costly than a false alarm (false positive)**.

We evaluate three operating points on the LR + SMOTE model:
- **Default (0.50)** — balanced trade-off, standard threshold
- **Max-F1 (0.596)** — maximises the harmonic mean of precision and recall
- **~90% Recall (0.320)** — catches ≈90% of diabetic patients at the cost of more false alarms


In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_curve, f1_score, recall_score,
                              precision_score, average_precision_score)

X_train_smote = np.load('Data/X_train_smote.npy')
y_train_smote = np.load('Data/y_train_smote.npy')
X_test_scaled = np.load('Data/X_test_scaled.npy')
y_test        = np.load('Data/y_test.npy')

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_smote, y_train_smote)
y_proba = lr.predict_proba(X_test_scaled)[:, 1]

prec, rec, thresholds = precision_recall_curve(y_test, y_proba)
ap = average_precision_score(y_test, y_proba)

idx_90    = np.argmin(np.abs(rec[:-1] - 0.90))
thresh_90 = float(thresholds[idx_90])
f1_scores = 2*prec[:-1]*rec[:-1]/(prec[:-1]+rec[:-1]+1e-9)
idx_f1    = np.argmax(f1_scores)
thresh_f1 = float(thresholds[idx_f1])

def metrics_at(t):
    yp = (y_proba >= t).astype(int)
    return (round(f1_score(y_test, yp), 4),
            round(precision_score(y_test, yp), 4),
            round(recall_score(y_test, yp), 4))

f1_d,  pr_d,  re_d  = metrics_at(0.50)
f1_f,  pr_f,  re_f  = metrics_at(thresh_f1)
f1_90, pr_90, re_90 = metrics_at(thresh_90)

print('Threshold Analysis — LR + SMOTE')
print(f'  Default  (0.50):               F1={f1_d},  Prec={pr_d},  Recall={re_d}')
print(f'  Max-F1   ({thresh_f1:.3f}): F1={f1_f},  Prec={pr_f},  Recall={re_f}')
print(f'  ~90% Rec ({thresh_90:.3f}): F1={f1_90}, Prec={pr_90}, Recall={re_90}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rec, prec, color='#2196F3', linewidth=2, label=f'LR+SMOTE (AP={ap:.3f})')
axes[0].axhline(y_test.mean(), color='gray', linestyle='--', linewidth=1, label='Random baseline')
axes[0].scatter(re_d,  pr_d,  color='black', s=80, zorder=5, label=f'Default t=0.50  R={re_d}')
axes[0].scatter(re_f,  pr_f,  color='green', s=80, zorder=5, label=f'Max-F1 t={thresh_f1:.2f}  R={re_f}')
axes[0].scatter(re_90, pr_90, color='red',   s=80, zorder=5, label=f'~90% Recall t={thresh_90:.2f}  R={re_90}')
axes[0].set_xlabel('Recall', fontsize=12); axes[0].set_ylabel('Precision', fontsize=12)
axes[0].set_title('Precision-Recall Curve — LR + SMOTE', fontsize=13)
axes[0].legend(fontsize=8); axes[0].set_xlim([0,1]); axes[0].set_ylim([0,1])

t_range = np.linspace(0.01, 0.99, 200)
f1_r, re_r, pr_r = [], [], []
for t in t_range:
    yp = (y_proba >= t).astype(int)
    if yp.sum() == 0:
        f1_r.append(0); re_r.append(0); pr_r.append(0)
    else:
        f1_r.append(f1_score(y_test, yp))
        re_r.append(recall_score(y_test, yp))
        pr_r.append(precision_score(y_test, yp, zero_division=0))

axes[1].plot(t_range, f1_r, color='#2196F3', linewidth=2, label='F1-score')
axes[1].plot(t_range, re_r, color='#FF5722', linewidth=2, label='Recall')
axes[1].plot(t_range, pr_r, color='#4CAF50', linewidth=2, label='Precision')
axes[1].axvline(0.50,      color='black', linestyle='--', linewidth=1, alpha=0.6, label='Default (0.50)')
axes[1].axvline(thresh_f1, color='green', linestyle='--', linewidth=1, alpha=0.8, label=f'Max-F1 ({thresh_f1:.2f})')
axes[1].axvline(thresh_90, color='red',   linestyle='--', linewidth=1, alpha=0.8, label=f'~90% Recall ({thresh_90:.2f})')
axes[1].set_xlabel('Decision Threshold', fontsize=12); axes[1].set_ylabel('Score', fontsize=12)
axes[1].set_title('Metrics vs Decision Threshold', fontsize=13)
axes[1].legend(fontsize=8); axes[1].set_xlim([0,1]); axes[1].set_ylim([0,1])

plt.suptitle('Threshold Optimisation — Logistic Regression + SMOTE', fontsize=13)
plt.tight_layout()
plt.savefig('Data/threshold_optimisation.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: Data/threshold_optimisation.png')


### Threshold Summary

| Threshold | F1 | Precision | Recall | Use Case |
|-----------|-----|-----------|--------|----------|
| 0.50 (default) | 0.4702 | 0.3409 | 0.7578 | General balanced use |
| 0.596 (max-F1) | 0.4817 | 0.3823 | 0.6509 | When precision matters more |
| 0.320 (~90% recall) | 0.4176 | 0.2719 | 0.9001 | **Screening — minimise missed cases** |

**Key insight**: lowering the threshold from 0.50 → 0.32 increases recall from 75.8% to 90.0%, catching 19% more diabetic patients, at the cost of roughly 20% more false positives. In a real screening context where patients are subsequently examined by a doctor, this tradeoff is clearly worthwhile.


## 5-Fold Cross-Validation

To verify that our reported test-set results are not due to a lucky train/test split, we run 5-fold stratified cross-validation on the training set.

- **LR + SMOTE**: SMOTE is applied *inside each fold* (only on the training portion) to avoid data leakage.
- **XGBoost scale_pos_weight**: Uses raw (unscaled) data consistently across train and validation splits.


In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

X_train_scaled = np.load('Data/X_train_scaled.npy')
X_train        = np.load('Data/X_train.npy')
y_train        = np.load('Data/y_train.npy')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_f1, lr_rec, lr_pre, lr_auc     = [], [], [], []
xgb_f1, xgb_rec, xgb_pre, xgb_auc = [], [], [], []

print('Running 5-fold CV (sequential, no multiprocessing)...')
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    print(f'  Fold {fold+1}/5 ...')

    # LR + SMOTE (on scaled data)
    X_tr_s, y_tr = X_train_scaled[tr_idx], y_train[tr_idx]
    X_val_s       = X_train_scaled[val_idx]
    y_val         = y_train[val_idx]
    X_sm, y_sm = SMOTE(random_state=42).fit_resample(X_tr_s, y_tr)
    lr_cv = LogisticRegression(max_iter=1000)
    lr_cv.fit(X_sm, y_sm)
    yp  = lr_cv.predict(X_val_s)
    ypr = lr_cv.predict_proba(X_val_s)[:,1]
    lr_f1.append(f1_score(y_val, yp))
    lr_rec.append(recall_score(y_val, yp))
    lr_pre.append(precision_score(y_val, yp))
    lr_auc.append(roc_auc_score(y_val, ypr))

    # XGBoost scale_pos_weight (on raw unscaled data)
    X_tr_r  = X_train[tr_idx]
    X_val_r = X_train[val_idx]
    n_n, n_p = np.bincount(y_tr.astype(int))
    xgb_cv = XGBClassifier(n_estimators=100, scale_pos_weight=n_n/n_p,
                            random_state=42, eval_metric='logloss')
    xgb_cv.fit(X_tr_r, y_tr)
    yp2  = xgb_cv.predict(X_val_r)
    ypr2 = xgb_cv.predict_proba(X_val_r)[:,1]
    xgb_f1.append(f1_score(y_val, yp2))
    xgb_rec.append(recall_score(y_val, yp2))
    xgb_pre.append(precision_score(y_val, yp2))
    xgb_auc.append(roc_auc_score(y_val, ypr2))

print('\n--- 5-Fold CV Results ---')
print(f'LR + SMOTE:')
print(f'  F1:        {np.mean(lr_f1):.4f} ± {np.std(lr_f1):.4f}  folds={[round(x,4) for x in lr_f1]}')
print(f'  Recall:    {np.mean(lr_rec):.4f} ± {np.std(lr_rec):.4f}')
print(f'  Precision: {np.mean(lr_pre):.4f} ± {np.std(lr_pre):.4f}')
print(f'  AUC-ROC:   {np.mean(lr_auc):.4f} ± {np.std(lr_auc):.4f}')
print(f'\nXGBoost scale_pos_weight:')
print(f'  F1:        {np.mean(xgb_f1):.4f} ± {np.std(xgb_f1):.4f}  folds={[round(x,4) for x in xgb_f1]}')
print(f'  Recall:    {np.mean(xgb_rec):.4f} ± {np.std(xgb_rec):.4f}')
print(f'  Precision: {np.mean(xgb_pre):.4f} ± {np.std(xgb_pre):.4f}')
print(f'  AUC-ROC:   {np.mean(xgb_auc):.4f} ± {np.std(xgb_auc):.4f}')


### Cross-Validation Results

| Model | CV F1 (mean ± std) | CV Recall | CV AUC-ROC |
|-------|-------------------|-----------|------------|
| LR + SMOTE | 0.4690 ± 0.0032 | 0.7587 ± 0.0083 | 0.8169 ± 0.0022 |
| XGBoost scale_pos_weight | 0.4682 ± 0.0028 | 0.7667 ± 0.0078 | 0.8182 ± 0.0025 |

**Key findings:**
- Both models are **remarkably stable** across folds (std < 0.004 for F1) — results are not due to a lucky split.
- CV F1 scores (0.469/0.468) closely match the held-out test set scores (0.47), confirming no overfitting.
- LR and XGBoost are effectively **tied** in performance — the simpler LR model is equally competitive.
- XGBoost has marginally higher recall (0.767 vs 0.759), suggesting it catches slightly more positive cases.


## Final Model Comparison

### Complete Results — All Models & Balancing Strategies

All models were evaluated on the same held-out test set (50,736 samples, 84/16 class split). The table below is the definitive comparison across every configuration tested in this project.

| Model | Training Data | F1 | AUC-ROC | Precision | Recall |
|---|---|---|---|---|---|
| Random Forest | Raw | 0.30 | 0.79 | 0.52 | 0.21 |
| XGBoost | Raw | 0.30 | 0.82 | 0.57 | 0.21 |
| Logistic Regression | Raw | 0.28 | 0.82 | 0.55 | 0.19 |
| KNN (k=5) | Raw | 0.31 | 0.73 | 0.43 | 0.24 |
| Random Forest | `class_weight='balanced'` | 0.28 | 0.79 | 0.49 | 0.20 |
| Random Forest | SMOTE | 0.37 | 0.79 | 0.47 | 0.31 |
| **XGBoost** | **`scale_pos_weight`** | **0.47** | **0.82** | 0.33 | **0.76** |
| XGBoost | SMOTE | 0.33 | 0.82 | 0.57 | 0.23 |
| **Logistic Regression** | **SMOTE** | **0.47** | **0.82** | 0.34 | **0.76** |
| Logistic Regression | Undersampling | 0.47 | 0.82 | 0.34 | 0.76 |
| KNN (k=5) | Undersampling | 0.43 | 0.76 | 0.30 | 0.74 |

**Joint best models: Logistic Regression + SMOTE and XGBoost + `scale_pos_weight`** — both F1 = 0.47, Recall = 0.76, AUC-ROC = 0.82.

---

### Why LR and XGBoost Perform Best

**Logistic Regression + SMOTE:** After scaling, the strongest risk factors (GenHlth, BMI, Age, HighBP) have roughly monotonic relationships with diabetes risk — exactly the structure a linear model captures well. SMOTE provides balanced training signal so the decision boundary shifts appropriately toward the minority class. The result is a globally stable boundary that generalises cleanly.

**XGBoost + `scale_pos_weight`:** XGBoost adjusts the loss function during gradient boosting to penalise misclassified diabetic patients more heavily (`scale_pos_weight ≈ 5.3`). This is mathematically equivalent to oversampling the minority class within the tree-building process. XGBoost's ability to model non-linear feature interactions explains why it also captures Income and Education as stronger drivers than LR does.

Both approaches address the same root cause — the imbalanced loss signal — but from different directions (data-level vs algorithm-level). Their near-identical final performance validates that the imbalance, not the model architecture, is the critical factor.

---

### Why Random Forest Struggles Despite Class Weighting

Random Forest with `class_weight='balanced'` only achieved F1 = 0.28, Recall = 0.20 — barely better than the raw baseline.

This is a known limitation of how RF implements class weighting. RF builds each tree using bootstrap sampling, and `class_weight='balanced'` adjusts sample weights within each tree. However, because each tree only sees a random subset of features and a bootstrapped sample, the weight adjustments are diluted across the ensemble. The majority class still dominates the vote aggregation.

In contrast, SMOTE physically rebalances the training data *before* RF sees it — every tree is trained on a genuinely balanced dataset. This is why RF + SMOTE (F1 = 0.37, Recall = 0.31) substantially outperforms RF + `class_weight='balanced'`, though it still falls short of LR and XGBoost, which have more effective built-in mechanisms for linearity and gradient-based imbalance correction respectively.

---

### Recall vs Precision Tradeoff

| Scenario | Best choice |
|---|---|
| Maximise recall (catch as many diabetics as possible) | LR + SMOTE or XGBoost `scale_pos_weight` at lowered threshold |
| Maximise F1 (balance precision and recall) | LR + SMOTE at threshold 0.596 (F1 = 0.48) |
| Maximise AUC-ROC (best probability ranking) | XGBoost + SMOTE (0.821) or XGBoost `scale_pos_weight` (0.818) |

For a diabetes **screening** tool, recall takes priority — the cost of a missed diagnosis (untreated disease progression) far outweighs the cost of a false positive (an unnecessary follow-up test). Threshold optimisation can push recall to 90%+ at the cost of lower precision — see the Threshold Optimisation section.

## Final Conclusions & Discussion

---

### 1. Central Finding

The dominant result of this project is unambiguous: **class balancing determines performance, not model complexity.**

Every model — including the powerful Random Forest and XGBoost ensembles — converges to recall ≈ 0.21 when trained on raw imbalanced data. The 84/16 class split causes all models to optimise for the majority class by default. Once balancing is applied (via SMOTE, undersampling, or loss-function weighting), recall jumps to 0.74–0.76 across model families.

The two best configurations are:
- **Logistic Regression + SMOTE** — F1 = 0.47, Recall = 0.76, AUC-ROC = 0.82
- **XGBoost + `scale_pos_weight`** — F1 = 0.47, Recall = 0.76, AUC-ROC = 0.82

---

### 2. Experiment 1 → Experiment 2 Connection

Logistic Regression + SMOTE was selected for SHAP analysis because it achieved the highest F1, is well-suited to `LinearExplainer` (exact Shapley values, no sampling), and its linear structure makes feature contributions directly interpretable.

SHAP confirmed the model learned genuine medical signal: **GenHlth, BMI, Age, and HighBP** are the top four drivers — consistent across both LR and XGBoost, and aligned with EDA findings. This cross-model agreement across two fundamentally different architectures provides strong evidence that these features are the true biological drivers of diabetes risk in this population.

---

### 3. Threshold Optimisation Summary

The default 0.5 threshold is not optimal for a screening use case. Key thresholds for LR + SMOTE:

| Threshold | F1 | Precision | Recall |
|---|---|---|---|
| 0.50 (default) | 0.4702 | 0.3409 | 0.7578 |
| 0.596 (max F1) | 0.4817 | 0.3823 | 0.6509 |
| 0.320 (~90% recall) | 0.4176 | 0.2719 | 0.9001 |

At threshold 0.32, the model catches 90% of diabetic patients — the appropriate operating point for a population-level screening tool.

---

### 4. Limitations

**Class imbalance:** Despite balancing, precision remains low (~0.27–0.34), meaning a significant proportion of flagged patients are false positives. Threshold calibration for a specific clinical setting is essential before deployment.

**SMOTE assumptions:** SMOTE interpolates linearly between minority-class samples. For a heterogeneous condition like diabetes with mixed binary and ordinal features, some synthetic patients may fall in implausible regions of feature space.

**Random Forest and class weighting:** RF's `class_weight='balanced'` was largely ineffective (Recall = 0.20), demonstrating that internal weighting is insufficient when the imbalance is severe. SMOTE — which rebalances the data before training — is more robust for RF.

**KNN scalability:** KNN prediction required ~10 minutes for 50k samples against a 200k training set. It is not viable in production at this scale without approximate nearest-neighbour methods.

**Self-reported data:** The BRFSS dataset relies on survey responses, introducing measurement error and response bias. Results may not generalise to clinical settings with laboratory data.

---

### 5. Future Work

- **Balanced RF/XGBoost with hyperparameter tuning** — Grid search over tree depth, learning rate, and number of estimators could close the gap between RF and the best models
- **SMOTE-ENN** — With sufficient compute, SMOTE-ENN's boundary cleaning may improve precision without sacrificing recall
- **Stacking/ensembling** — A meta-learner combining LR + SMOTE and XGBoost + `scale_pos_weight` predictions could outperform either individually
- **Real-world validation** — Prospective validation on a clinical cohort with confirmed diagnoses and laboratory values is the necessary next step before deployment